In [1]:
import pandas as pd, textwrap

In [2]:
arts = pd.read_feather('../candidate_public/candidate_data/articles.f')

arts.head()
arts.info()

print(f'Статьи: строк = {len(arts)}, колонки = {list(arts.columns)}')

<class 'pandas.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   article_id  793 non-null    int64 
 1   title       793 non-null    string
 2   body        793 non-null    string
dtypes: int64(1), string(2)
memory usage: 9.7 MB
Статьи: строк = 793, колонки = ['article_id', 'title', 'body']


/home/d_tsyp/projects/support-article-ranking/.venv/lib/python3.14/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [3]:
print("Пример заголовков: ")
for t in arts.title.head(10):
    print(" ", t)

Пример заголовков: 
  Имя или название компании
  Понять, что профиль заблокирован
  Не допустить блокировки профиля
  Оставить или удалить профиль
  Удалить профиль
  Когда не получится удалить профиль
  Профиль удалён
  Квитанция отображается некорректно
  Позвонить продавцу
  Открыть контакты в резюме


In [4]:
print("Длина body по статьям: ")
d = arts.body.str.len().describe()
print(f"минимум={int(d["min"])}, медиана = {int(d["50%"])}, максимум = {int(d["max"])}")

Длина body по статьям: 
минимум=31, медиана = 3513, максимум = 901357


In [5]:
row = arts[arts.article_id==1909].iloc[0]
print("article_id =", row.article_id)
print("title=", row.title)
print("body, сырой HTML: ")
print(row.body[:1000])

article_id = 1909
title= Отправить заказ
body, сырой HTML: 
<ol><li><p>У вас будет <strong>2 рабочих дня, </strong>чтобы отправить товар<strong> </strong>(не считая день оплаты заказа).</p><p>Если не успеваете — продлите отправку на 4 календарных дня по кнопке <em>Продлить срок отправки </em>на странице заказа.</p><p><a href="https://support.avito.ru/articles/1918">Что делать, если не успеваете отправить заказ</a>.</p></li><li><p>Проверьте, что товар подходит под <strong>разрешённые габариты</strong> нужной службы доставки. Если это не так — вещь не примут.</p><p><a href="https://support.avito.ru/articles/4328">Посмотреть разрешённые габариты и вес</a>.</p></li><li><p>Перед отправкой <strong>товар нужно надёжно упаковать</strong> — это можно сделать дома или в пункте приёма.</p><p><a href="https://support.avito.ru/articles/1907">Не во всех пунктах есть бесплатная упаковка</a>.</p><div class="factoid factoid_disclaimer"><p>Надёжная упаковка — обязательное условие, без неё заказ не приму

In [6]:
cal = pd.read_feather("../candidate_public/candidate_data/calibration.f")

print(f"Запросы calibration: строк = {len(cal)}, колонки = {list(cal.columns)}")
print("Живые примеры (текст запроса -> правильные article_id):")

for _, r in cal.head(6).iterrows():
    print(f"Q: {r.query_text}")
    print(f"gt: {r.ground_truth}")

# распределение числа правильных статей
n = cal.ground_truth.str.split().apply(len)

print(f"Сколько правильных статей на один запрос: {n.value_counts().sort_index().to_dict()}")
print(f"(у {(n>=2).mean()*100:.0f}% запросов их 2 и больше)")

Запросы calibration: строк = 500, колонки = ['query_id', 'query_text', 'ground_truth']
Живые примеры (текст запроса -> правильные article_id):
Q: Как передать товар через службу авито
gt: 1909 4234
Q: Можете подсказать, если заказать товар Авито доставкой и оплатить также через авито, в случае возврата товара деньги безусловно мне возвращаются?
gt: 2865 4400
Q: Здравствуйте. Как отправить товар через Авито.
gt: 1909
Q: как получить деньги за возрат если продавец уже забрал товар?
gt: 4400 4403
Q: Когда мне прийдут деньги за доставку, сегодня уже <DATE>
gt: 4361
Q: Почему пишете что доставка <MONEY> а когда оплачиваешь получается больше <MONEY>?
gt: 1951
Сколько правильных статей на один запрос: {1: 279, 2: 182, 3: 38, 4: 1}
(у 44% запросов их 2 и больше)


/home/d_tsyp/projects/support-article-ranking/.venv/lib/python3.14/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [7]:
id2t = dict(zip(arts.article_id, arts.title))
print("Запрос: «как получить деньги за возрат если продавец уже забрал товар?»")
print("Правильные статьи:")
for a in [4400, 4403]:
    print(f"{a}: {id2t[a]}")
print("Запрос: «Почему пишете что доставка <MONEY> а когда оплачиваешь получается больше?»")
for a in [1951]:
    print(f"{a}: {id2t[a]}")
print("Запрос: «Как передать товар через службу авито»")
for a in [1909, 4234]:
    print(f"{a}: {id2t[a]}")

Запрос: «как получить деньги за возрат если продавец уже забрал товар?»
Правильные статьи:
4400: Покупателю — отказаться от товара или вернуть его
4403: Продавцу — товар не забирают или вернули
Запрос: «Почему пишете что доставка <MONEY> а когда оплачиваешь получается больше?»
1951: Кто оплачивает доставку и сколько она стоит
Запрос: «Как передать товар через службу авито»
1909: Отправить заказ
4234: Как продавать и покупать с доставкой


Видно, что пользователь и автор статьи описывают одно и то же разными словами. Так же есть опечатки, нужно чистить HTML теги и заглушки вроде MONEY, DATE и т.д.

### Выводы:
1. Надо чистить разметку, иначе поиск захламлён тегами
2. Есть опечатки/разговорный стиль и т.д. Простой поиск заберет примерно половину, остальное нужно думать - LSA, эмбеддинги.
3. Почти у половины запросов несколько правильных статей

In [8]:
from text_processing import clean_html, tokenize

body = arts[arts.article_id==1909].body.iloc[0]

print(f"До (сырой HTML): {body[:280]}")

clean = clean_html(body)

print(f"После (чистый текст): {clean[:280]}")

print("Токены запроса «Как отправить товар через Авито доставкой» (stem=False):")
print(" ", tokenize("Как отправить товар через Авито доставкой", stem=False))
print("Те же токены со стеммингом (stem=True):")
print(" ", tokenize("Как отправить товар через Авито доставкой", stem=True))

До (сырой HTML): <ol><li><p>У вас будет <strong>2 рабочих дня, </strong>чтобы отправить товар<strong> </strong>(не считая день оплаты заказа).</p><p>Если не успеваете — продлите отправку на 4 календарных дня по кнопке <em>Продлить срок отправки </em>на странице заказа.</p><p><a href="https://supp
После (чистый текст): У вас будет 2 рабочих дня, чтобы отправить товар (не считая день оплаты заказа). Если не успеваете — продлите отправку на 4 календарных дня по кнопке Продлить срок отправки на странице заказа. Что делать, если не успеваете отправить заказ . Проверьте, что товар подходит под разре
Токены запроса «Как отправить товар через Авито доставкой» (stem=False):
  ['отправить', 'товар', 'авито', 'доставкой']
Те же токены со стеммингом (stem=True):
  ['отправ', 'товар', 'авит', 'доставк']


In [9]:
import collections, numpy as np
from rank_bm25 import BM25Okapi
from metrics import map_at_k

gt = {r.query_id: {int(x) for x in r.ground_truth.split()} for r in cal.itertuples()}
ids = arts.article_id.tolist()

# Бейзлайн: всем один и тот же топ-10 популярных в calibration
cnt = collections.Counter(a for rel in gt.values() for a in rel)
top10 = [a for a,_ in cnt.most_common(10)]
base = {q: top10 for q in gt}
print(f"0. Бейзлайн топ-популярных всем: MAP@10 = {map_at_k(base, gt):.4f}")

def run_bm25(texts, stem):
    tok = lambda s: tokenize(s, stem=stem)
    bm = BM25Okapi([tok(t) for t in texts])
    preds = {}
    for r in cal.itertuples():
        sc = bm.get_scores(tok(r.query_text))
        preds[r.query_id] = [ids[i] for i in np.argsort(sc)[::-1][:10]]
    return map_at_k(preds, gt)

raw = arts.body
clean = arts.body.map(clean_html)
print(f"1. BM25 по сырому HTML (no stem): MAP@10 = {run_bm25(raw, False):.4f}")
print(f"2. BM25 по чистому тексту (no stem): MAP@10 = {run_bm25(clean, False):.4f}")
print(f"3. BM25 по чистому тексту (со стеммингом): MAP@10 = {run_bm25(clean, True):.4f}")

0. Бейзлайн топ-популярных всем: MAP@10 = 0.3176
1. BM25 по сырому HTML (no stem): MAP@10 = 0.2844
2. BM25 по чистому тексту (no stem): MAP@10 = 0.2985
3. BM25 по чистому тексту (со стеммингом): MAP@10 = 0.2546


1. Очистка HTML помогла
2. Стемминг сделал хуже
3. BM25 хуже базового бейзлайна (что странно) - надо смотреть где он ошибается

In [10]:
from metrics import ap_at_k

clean = arts.body.map(clean_html)

tok = lambda s: tokenize(s, stem=False)
bm = BM25Okapi([tok(t) for t in clean])

rows=[]
for r in cal.itertuples():
    sc = bm.get_scores(tok(r.query_text))
    pred = [ids[i] for i in np.argsort(sc)[::-1][:10]]
    ap = ap_at_k(pred, gt[r.query_id])
    rows.append((r.query_id, r.query_text, gt[r.query_id], pred, ap))

df = pd.DataFrame(rows, columns=['qid','q','gt','pred','ap'])
print("Распределение AP@10 по 500 запросам:")
print(f"AP = 0.00 (полный промах): {(df.ap==0).sum()} запросов ({(df.ap==0).mean()*100:.0f}%)")
print(f"0 < AP < 0.5 (слабо): {((df.ap>0)&(df.ap<0.5)).sum()}")
print(f"AP >= 0.5 (хорошо): {(df.ap>=0.5).sum()}")
print(f"AP = 1.0 (идеально): {(df.ap==1).sum()}")
print(f"средний AP (=MAP@10): {df.ap.mean():.4f}")
df.to_pickle('/tmp/bm25_errors.pkl')

Распределение AP@10 по 500 запросам:
AP = 0.00 (полный промах): 167 запросов (33%)
0 < AP < 0.5 (слабо): 193
AP >= 0.5 (хорошо): 140
AP = 1.0 (идеально): 64
средний AP (=MAP@10): 0.2985


In [12]:
miss = df[df.ap==0].reset_index(drop=True)
rng = np.random.RandomState(3)
idx = rng.choice(len(miss), 6, replace=False)
for k in idx:
    r = miss.iloc[k]
    print(f"Запрос: {str(r['q'])[:140]}")
    print('Надо:', ' | '.join('%s: %s'%(a,id2t[a]) for a in r['gt']))
    print('BM25:', ' | '.join('%s: %s'%(a,id2t[a]) for a in r['pred'][:3]))

Запрос: здравствуйте !сделал заказ,вернул его,но деньги не пришли
Надо: 4384: Баланс для покупок | 4219: Покупателю
BM25: 3006: Оценить покупателя | 4361: Продавцу | 2865: Когда вернутся деньги за доставку
Запрос: Здравствуйте, возможно ли отправить Авито доставкой бампера автомобиля?
Надо: 4328: Что можно заказать и отправить
BM25: 4401: Авто в кредит | 4232: Автотека | 2253: Объявление исчезло из поиска
Запрос: здравствуйте ! не могу разместить объявление ? выдает что ошибка
Надо: 2222: Ошибка при подаче объявления через приложение
BM25: 4008: Не получается опубликовать объявление | 4307: Лимит бесплатных размещений | 3777: Транспорт
Запрос: добрый день, у меня 200 бонусов но почему то они не применяются когда пытаюсь оформить покупку
Надо: 4219: Покупателю | 2646: Оплата заказов с доставкой
BM25: 4424: Бонусы Авито | 4395: Бонусы | 4222: Запланировать услуги продвижения
Запрос: Как отправить безопастно ювелирное изделие?
Надо: 1907: Упаковать товар
BM25: 4320: Как поддерживать досто

Увидели ошибки, теперь понятно. Типы ошибок:
1. Смысловой разрыв
2. Редкие слова уводят в сторону («отправить Авито доставкой бампера автомобиля». Правильная — «Что можно заказать и отправить» (вопрос-то «можно ли это вообще отправить?»). А BM25 вцепился в редкие слова «автомобиль/бампер» и выдал «Авто в кредит», «Автотека».)
3. Статьи близнецы и BM25 берёт не ту ( «не могу разместить объявление, выдаёт ошибку» → надо 2222 «Ошибка при подаче объявления через приложение». BM25 выдал 4008 «Не получается опубликовать объявление» — почти тот же смысл, но по разметке это не тот ответ. В базе есть статьи двойники, и модель путается между ними)
4. Ложный лексический мэтч («200 бонусов не применяются при покупке». BM25 выдал «Бонусы Авито», «Бонусы» — идеальное совпадение слова, но правильные ответы «Покупателю» и «Оплата заказов с доставкой»)

In [13]:
# токены каждой статьи (title+body)
art_tokens = {}
for r in arts.itertuples():
    art_tokens[r.article_id] = set(tokenize(r.title + ' ' + clean_html(r.body), stem=False))

no_overlap = 0
for r in miss.itertuples():
    qt = set(tokenize(r.q, stem=False))
    # есть ли хоть одно общее слово с любой из правильных статей
    shares = any(qt & art_tokens[a] for a in r.gt)
    if not shares:
        no_overlap += 1
print(f"Полных промахов (AP=0): {len(miss)}")
print(f"из них без единого общего слова с правильной статьёй: {no_overlap} ({no_overlap/len(miss)*100:.0f}%)")
# сколько промахов, где общее слово есть, но статья всё равно не поднялась (типы 2/3/4)
print(f'промахов, где слова общие есть, но статья не в топ-10: {len(miss)-no_overlap} ({(len(miss)-no_overlap)/len(miss)*100:.0f}%)')

Полных промахов (AP=0): 167
из них без единого общего слова с правильной статьёй: 6 (4%)
промахов, где слова общие есть, но статья не в топ-10: 161 (96%)


In [14]:
miss_qids = set(df[df.ap==0].qid)

# для промахов: на какой позиции стоит лучшая из правильных статей
best_ranks=[]
for r in cal.itertuples():
    if r.query_id not in miss_qids: continue
    sc = bm.get_scores(tok(r.query_text))
    order = np.argsort(sc)[::-1]
    rankpos = {ids[order[p]]: p for p in range(len(order))}
    best = min(rankpos[a] for a in gt[r.query_id])  # 0-индекс
    best_ranks.append(best+1)  # в 1-индекс
best_ranks=np.array(best_ranks)
print("Где реально стоит лучшая правильная статья у 167 промахов:")
for lo,hi in [(11,20),(21,50),(51,100),(101,793)]:
    c=((best_ranks>=lo)&(best_ranks<=hi)).sum()
    print(f"позиции {lo:>3}-{hi:<3}: {c:3} запросов ({c/len(best_ranks)*100:.0f}%)")
print(f"медиана позиции: {int(np.median(best_ranks))}")

Где реально стоит лучшая правильная статья у 167 промахов:
позиции  11-20 :  78 запросов (47%)
позиции  21-50 :  57 запросов (34%)
позиции  51-100:  16 запросов (10%)
позиции 101-793:  16 запросов (10%)
медиана позиции: 23


BM25 почти всегда находит правильную статью и держит её приблизительно в топ-50, часто на 11–20 месте. Он просто не дотягивает её до топ-10. Те он плохой финальный сортировщик

Таким образом нужна двухэтапная схема поиска:
1. BM25 быстро отбирает пускай топ-50 статей кандидатов. И почти наверняка правильная уже среди них (в 81% промахов в топ-50).
2. Переранжирование. Нужен второй сигнал (основанный на смысле) который пересматривает эти 50 кандидатов и поднимает настоящие ответы в топ-10